In [1]:
!pip install --quiet google-cloud-pubsub

import google.auth
from google.cloud import bigquery

PROJECT_ID = "qwiklabs-gcp-00-f469c11ef7ce"
DATASET = "flight_stream"
TABLE = "transponder_msgs"
PUBSUB_PROJECT = "paul-leroy"
TOPIC = "flight-transponder"
SUB_ID = "flight-transponder-sub"

credentials, _ = google.auth.default()
client = bigquery.Client(project=PROJECT_ID, credentials=credentials, location="US")
print("connected")

connected


In [2]:
client.query(f"CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.{DATASET}` OPTIONS(location='US')").result()

schema = [
    bigquery.SchemaField("MT",   "STRING", description="Message type"),
    bigquery.SchemaField("TT",   "INT64",  description="Transmission type 1-8"),
    bigquery.SchemaField("SID",  "STRING", description="Session record number"),
    bigquery.SchemaField("AID",  "STRING", description="Aircraft record number"),
    bigquery.SchemaField("Hex",  "STRING", description="Aircraft Mode S hex code"),
    bigquery.SchemaField("FID",  "STRING", description="Flight record number"),
    bigquery.SchemaField("DMG",  "DATE",   description="Date message generated"),
    bigquery.SchemaField("TMG",  "TIME",   description="Time message generated"),
    bigquery.SchemaField("DML",  "DATE",   description="Date message logged"),
    bigquery.SchemaField("TML",  "TIME",   description="Time message logged"),
    bigquery.SchemaField("CS",   "STRING", description="Callsign"),
    bigquery.SchemaField("Alt",  "INT64",  description="Altitude (flight level)"),
    bigquery.SchemaField("GS",   "INT64",  description="Ground speed"),
    bigquery.SchemaField("Trk",  "INT64",  description="Track"),
    bigquery.SchemaField("Lat",  "FLOAT64",description="Latitude"),
    bigquery.SchemaField("Lng",  "FLOAT64",description="Longitude"),
    bigquery.SchemaField("VR",   "INT64",  description="Vertical rate"),
    bigquery.SchemaField("Sq",   "STRING", description="Squawk code"),
    bigquery.SchemaField("Alrt", "INT64",  description="Squawk change flag"),
    bigquery.SchemaField("Emer", "INT64",  description="Emergency flag"),
    bigquery.SchemaField("SPI",  "INT64",  description="Ident flag"),
    bigquery.SchemaField("Gnd",  "INT64",  description="Ground squat flag"),
]

table_id = f"{PROJECT_ID}.{DATASET}.{TABLE}"
table_obj = client.create_table(bigquery.Table(table_id, schema=schema), exists_ok=True)
print(f"Table ready: {table_id} ({len(table_obj.schema)} fields)")

Table ready: qwiklabs-gcp-00-f469c11ef7ce.flight_stream.transponder_msgs (22 fields)


In [3]:
from google.cloud import pubsub_v1

subscriber = pubsub_v1.SubscriberClient(credentials=credentials)
topic_path = f"projects/{PUBSUB_PROJECT}/topics/{TOPIC}"
sub_path = subscriber.subscription_path(PROJECT_ID, SUB_ID)

try:
    subscriber.create_subscription(request={"name": sub_path, "topic": topic_path})
    print(f"Subscription created: {sub_path}")
except Exception as e:
    print(f"Note: {type(e).__name__}: {e}")
    print("(If it says 'already exists', that's fine — we'll reuse it.)")

Subscription created: projects/qwiklabs-gcp-00-f469c11ef7ce/subscriptions/flight-transponder-sub


In [4]:
import time

FIELD_NAMES = ["MT","TT","SID","AID","Hex","FID","DMG","TMG","DML","TML",
               "CS","Alt","GS","Trk","Lat","Lng","VR","Sq","Alrt","Emer","SPI","Gnd"]
INT_FIELDS   = {"TT","Alt","GS","Trk","VR","Alrt","Emer","SPI","Gnd"}
FLOAT_FIELDS = {"Lat","Lng"}
DATE_FIELDS  = {"DMG","DML"}

def parse_line(line):
    parts = line.split(",")
    if len(parts) < 22:
        parts += [""] * (22 - len(parts))
    row = {}
    for name, val in zip(FIELD_NAMES, parts[:22]):
        val = val.strip()
        if not val:
            continue                       # empty -> leave NULL
        if name in INT_FIELDS:
            try: row[name] = int(val)
            except ValueError: pass
        elif name in FLOAT_FIELDS:
            try: row[name] = float(val)
            except ValueError: pass
        elif name in DATE_FIELDS:
            row[name] = val.replace("/", "-")   # 2025/04/15 -> 2025-04-15
        else:
            row[name] = val                      # STRING + TIME fields
    return row

table_id = f"{PROJECT_ID}.{DATASET}.{TABLE}"
collected = 0
deadline = time.time() + 120   # ~2 minutes

print("Collecting flight messages for ~2 minutes...")
while time.time() < deadline:
    resp = subscriber.pull(request={"subscription": sub_path, "max_messages": 200}, timeout=20)
    if not resp.received_messages:
        continue
    rows, ack_ids = [], []
    for m in resp.received_messages:
        line = m.message.data.decode("utf-8").strip()
        if line:
            rows.append(parse_line(line))
        ack_ids.append(m.ack_id)
    if rows:
        errors = client.insert_rows_json(table_id, rows)
        if errors:
            print("Insert errors (first):", errors[0])
        else:
            collected += len(rows)
    subscriber.acknowledge(request={"subscription": sub_path, "ack_ids": ack_ids})
    print(f"  collected: {collected}")

print(f"Done. Total inserted: {collected}")

  collected: 114
  collected: 214
  collected: 414
  collected: 614
  collected: 814
  collected: 1014
  collected: 1214
  collected: 1291
  collected: 1491
  collected: 1691
  collected: 1891
  collected: 2091
  collected: 2291
  collected: 2491
  collected: 2691
  collected: 2891
  collected: 3091
  collected: 3168
  collected: 3368
  collected: 3568
  collected: 3768
  collected: 3968
  collected: 4168
  collected: 4368
  collected: 4568
  collected: 4768
  collected: 4968
  collected: 5045
  collected: 5245
  collected: 5445
  collected: 5645
  collected: 5845
  collected: 6045
  collected: 6122
  collected: 6322
  collected: 6522
  collected: 6722
  collected: 6922
  collected: 7122
  collected: 7322
  collected: 7432
  collected: 7632
  collected: 7832
  collected: 8032
  collected: 8199
  collected: 8399
  collected: 8599
  collected: 8799
  collected: 8999
  collected: 9199
  collected: 9276
  collected: 9476
  collected: 9676
  collected: 9876
  collected: 10076
  collected: 1

In [5]:
client.query(f"SELECT COUNT(*) AS record_count FROM `{PROJECT_ID}.{DATASET}.{TABLE}`").to_dataframe()

,record_count
0,23312


In [6]:
locations = client.query(f"""
SELECT ST_GEOGPOINT(Lng, Lat) AS Location
FROM `{PROJECT_ID}.{DATASET}.{TABLE}`
WHERE Lat IS NOT NULL AND Lng IS NOT NULL
""").to_dataframe()
print(f"{len(locations)} messages have coordinates")
locations.head()

1224 messages have coordinates


,Location
0,POINT(-0.24511 51.56026)
1,POINT(-0.6543 51.40657)
2,POINT(-0.10482 51.75751)
3,POINT(-0.86159 51.23679)
4,POINT(-0.86035 51.73901)
